# Can one filter recognize a scratch after it moves?
**CS418 · Computer Vision · Lab 11 · Week of 23 November 2026**

A camera records small grayscale tiles. Each contains a horizontal (class 0) or vertical (class 1) scratch. You will train filters on images from one simulated camera, inspect a one-pixel translation and adapt the network to a second camera where the scratches appear dark.

Open this notebook in Jupyter or upload it to Google Colab. Run it from the top. Complete the functions marked **TODO** and write short explanations in the answer cells. Python, NumPy 1.20+ and Matplotlib are sufficient on a CPU. All tiles are generated here.

**120 minutes:** C1 18 min · C2 22 min · break 10 min · C3 18 min · C4 25 min · C5 27 min. The boundary experiment is optional.

## C1 · From a patch to a class probability
Every tile in the experiment has one channel and 12 × 12 pixels. To work by hand we first crop a 3 × 4 toy image. A receptive field is the set of input pixels used for one response. Here it is a 2 × 2 patch. Lay the kernel on a patch, multiply matching entries, add the products and the bias, then pass the response through ReLU. Use the kernel in its displayed orientation, the cross-correlation convention of CNN libraries. With stride 1 the window moves one pixel at a time. Without padding only fully contained windows count.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
np.set_printoptions(precision=4, suppress=True)

def make_tiles(n, seed, domain="source"):
    """Generate balanced batches of 12 x 12 tiles, class 0 horizontal and class 1 vertical."""
    rng = np.random.default_rng(seed)
    labels = np.arange(n) % 2
    rng.shuffle(labels)
    images = np.empty((n, 12, 12))
    for i, label in enumerate(labels):
        if domain == "source":
            half = int(rng.choice([2, 3]))
            row, col = rng.integers(4, 8, 2)
            tile = rng.normal(0.09, 0.035, (12, 12))
            change = rng.uniform(0.65, 0.9)
        else:
            half = int(rng.choice([2, 3]))
            row, col = rng.integers(3, 9, 2)
            tile = rng.normal(0.70, 0.075, (12, 12))
            change = -rng.uniform(0.40, 0.65)
        if label == 0:
            tile[row, col-half:col+half+1] += change
        else:
            tile[row-half:row+half+1, col] += change
        images[i] = np.clip(tile, 0, 1)
    return images, labels.astype(float)

def sigmoid(score):
    return 1 / (1 + np.exp(-np.clip(score, -60, 60)))

def bce(logits, labels):
    return np.mean(np.logaddexp(0, logits) - labels * logits)

source_train = make_tiles(96, 4181)
source_val = make_tiles(48, 4182)
target_train = make_tiles(24, 4183, "target")
target_val = make_tiles(48, 4184, "target")

fig, axes = plt.subplots(2, 4, figsize=(6.5, 3.6), constrained_layout=True)
for row, (name, data) in enumerate([("Camera A", source_train), ("Camera B", target_train)]):
    images, labels = data
    ids = np.r_[np.flatnonzero(labels == 0)[:2], np.flatnonzero(labels == 1)[:2]]
    for ax, idx in zip(axes[row], ids):
        ax.imshow(images[idx], cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(f"{name} · {'V' if labels[idx] else 'H'}", fontsize=9)
        ax.axis("off")
plt.show()

image = np.array([[0, 1, 0, 0], [1, 1, 0, 1], [0, 0, 1, 0]], dtype=float)
kernel = np.array([[1, -1], [0, 1]], dtype=float)
bias = -0.5
print("Toy image:\n", image, "\nKernel:\n", kernel)
print("Window (0,1):\n", image[:2, 1:3])

In [ ]:
def response_at(image, kernel, bias, row, col):
    height, width = kernel.shape
    # TODO: slice a patch starting at (row, col), combine it with the kernel and add bias.
    return None

In [ ]:
def toy_forward(image, kernel, bias):
    patches = sliding_window_view(image, kernel.shape)
    rows, cols = patches.shape[:2]
    pre = np.array([[response_at(image, kernel, bias, i, j) for j in range(cols)] for i in range(rows)])
    hidden = np.maximum(pre, 0)
    # The toy head has weight 1 on the mean hidden feature and a fixed bias of -0.2.
    logit = hidden.mean() - 0.2
    return patches, pre, hidden, logit


patch_ok = response_at(image, kernel, bias, 0, 0) is not None
if patch_ok:
    np.testing.assert_allclose(response_at(image, kernel, bias, 0, 0), -0.5)
    patches, pre, hidden, logit = toy_forward(image, kernel, bias)
    assert pre.shape == (2, 3)
    print("Pre-activation map:\n", pre)
    print("ReLU map:\n", hidden)
    print(f"Logit {logit:.5f}, P(vertical) {sigmoid(logit):.5f}, BCE for label 1: {bce(logit, 1):.5f}")
else:
    print("Complete response_at in C1 and rerun this cell.")

**Record your evidence**

Calculate the window starting at (0,1) by hand and name one position where ReLU passes zero onward. Which hidden activation is larger than 1? What does the probability refer to? What would two affine layers collapse to if you removed every nonlinearity between them?

*Your answer:*

## C2 · Several windows change one weight
Keep the toy head fixed. Set $y=1$ and use sigmoid binary cross-entropy. Its derivative with respect to the logit is $p-y$. The global mean splits that derivative between six locations, then the ReLU gate blocks inactive locations. If $a_{ij}$ is the pre-activation,
$$\delta_{ij}=(p-y)\,\mathbf{1}[a_{ij}>0]/6,\quad \frac{\partial L}{\partial K_{uv}}=\sum_{ij}\delta_{ij}X_{i+u,j+v}.$$
By hand, circle the locations with both an open gate and a nonzero top-left input pixel. Predict the sign of the update to $K_{00}$. Write **shared_gradient** to sum all contributions and the bias gradient. In code, two trailing singleton axes broadcast the map of derivatives across the two kernel axes.

In [ ]:
def shared_gradient(patches, delta):
    # patches shape (out_h, out_w, k_h, k_w), delta shape (out_h, out_w).
    # TODO: weight each patch by its delta. Sum over spatial positions for K.
    # Return (d_kernel, d_bias). Each output position uses the same bias.
    return None

In [ ]:
gradient_ok = shared_gradient(np.ones((1, 1, 2, 2)), np.ones((1, 1))) is not None
if patch_ok and gradient_ok:
    patches, pre, hidden, logit = toy_forward(image, kernel, bias)
    delta = (sigmoid(logit) - 1) * (pre > 0) / pre.size
    grad_k, grad_b = shared_gradient(patches, delta)
    # Two active windows have a nonzero top-left input in this example.
    np.testing.assert_allclose(grad_k[0, 0], 2 * (sigmoid(logit) - 1) / 6)
    np.testing.assert_allclose(grad_b, delta.sum())
    rate = 0.2
    updated_k, updated_b = kernel - rate * grad_k, bias - rate * grad_b
    new_logit = toy_forward(image, updated_k, updated_b)[-1]
    print("Contributions to K[0,0]:\n", delta * patches[:, :, 0, 0])
    print("Kernel gradient:\n", grad_k, "\nBias gradient:", round(grad_b, 5))
    print("Updated kernel:\n", updated_k)
    print("Old and new BCE:", np.round([bce(logit, 1), bce(new_logit, 1)], 5))
else:
    print("Complete C1 and shared_gradient in C2, then rerun.")

**Record your evidence**

Show the two terms for K[0,0], its updated value and the old/new loss. Which lines calculate gradients? Which lines change a parameter? What would a one-patch update miss?

*Your answer:*

## C3 · Build the spatial part of the network
Return to the camera tiles. The full model uses six trainable 3 × 3 kernels, stride 1, no padding (valid convolution), ReLU, 2 × 2 max pooling at stride 2, a mean over the remaining spatial positions, then one binary score. Tensor order for internal feature maps is **(batch, channels, height, width)**. The input helper stores the single grayscale plane as (batch, height, width).

Before running, predict the shapes after convolution, pooling and global mean. Count learnable numbers in the filter bank (including six biases) and in the binary head (including one bias). A max-pool window keeps its largest value. The global mean then returns one feature per channel.

In [ ]:
def output_width(side, kernel, stride, pad):
    # TODO: implement floor((side + 2*pad - kernel) / stride) + 1 for dilation 1.
    return None


def pool2x2(active):
    # active: N x F x H x W, with even H and W.
    n, f, h, w = active.shape
    blocks = active.reshape(n, f, h//2, 2, w//2, 2)
    # TODO: return (blocks, maxima). Reduce across the two within-window axes.
    return None

In [ ]:
tiny = np.array([[[[1, 3, 0, 2], [2, 0, 4, 1], [1, 1, 2, 3], [0, 2, 1, 0]]]], dtype=float)
pool_ok = pool2x2(tiny) is not None
size_ok = output_width(12, 3, 1, 0) is not None
if pool_ok:
    blocks, pooled = pool2x2(tiny)
    np.testing.assert_allclose(pooled, [[[[3, 4], [2, 3]]]])
    print("Toy pooled map:\n", pooled[0, 0])
if size_ok:
    assert output_width(12, 3, 1, 0) == 10 and output_width(10, 2, 2, 0) == 5
    print("Try the changed camera: 15 pixels, kernel 3, stride 2, padding 1. Spatial output:", output_width(15, 3, 2, 1))
if not (pool_ok and size_ok):
    print("Complete the C3 functions to check the geometry.")

**Record your evidence**

For one grayscale tile, give the full feature-map shapes, the number of local convolution responses, and the parameter total. Recalculate the filter-bank count if a new camera has three input channels. What changes if only the image size doubles?

*Your answer:*

## C4 · Train the filters and move the scratch
The supplied trainer performs the forward pass, backpropagation and updates with mini-batch SGD and momentum. Your pooling function is part of the model. There are 96 source training images and 48 validation images. Use the fixed schedule of 35 epochs. The threshold for vertical is logit ≥ 0, equivalent to probability ≥ 0.5. Read the **forward** routine once, then run the experiment.

In the second half, move every validation tile right by one pixel, filling the new column with zero. The source generator leaves space around the mark, so its orientation label is preserved. Compare original and moved responses **at corresponding coordinates** on the common interior. Then examine agreement of the output labels and the absolute change in probability.

In [ ]:
def initialize(seed=23, channels=6):
    rng = np.random.default_rng(seed)
    return {"K": rng.normal(0, 0.42, (channels, 3, 3)),
            "b": np.full(channels, 0.03),
            "v": rng.normal(0, 0.5, channels),
            "c": np.zeros(1)}

def forward(images, params):
    patches = sliding_window_view(images, params["K"].shape[-2:], axis=(1, 2))
    pre = np.einsum("nhwij,fij->nfhw", patches, params["K"]) + params["b"][None, :, None, None]
    active = np.maximum(pre, 0)
    blocks, pooled = pool2x2(active)
    features = pooled.mean(axis=(2, 3))
    logits = features @ params["v"] + params["c"][0]
    return logits, (patches, pre, blocks, pooled, features)

def backward(labels, logits, cache, params):
    patches, pre, blocks, pooled, features = cache
    delta = (sigmoid(logits) - labels) / len(labels)
    grads = {"v": features.T @ delta, "c": np.array([delta.sum()])}
    # Global mean divides the gradient equally among all pooled positions.
    pool_grad = delta[:, None, None, None] * params["v"][None, :, None, None]
    pool_grad = pool_grad / (pooled.shape[2] * pooled.shape[3])
    winners = blocks == pooled[:, :, :, None, :, None]
    # At a tie, share the gradient equally between tied maxima.
    shares = winners / winners.sum(axis=(3, 5), keepdims=True)
    active_grad = (shares * pool_grad[:, :, :, None, :, None]).reshape(pre.shape)
    pre_grad = active_grad * (pre > 0)
    grads["K"] = np.einsum("nfhw,nhwij->fij", pre_grad, patches)
    grads["b"] = pre_grad.sum(axis=(0, 2, 3))
    return grads

def measures(params, images, labels):
    logits, _ = forward(images, params)
    return bce(logits, labels), np.mean((logits >= 0) == labels)

def train_model(start, train_set, valid_set, trainable, epochs, rate, seed=44):
    # Copy all arrays, including for the frozen experiment.
    params = {name: value.copy() for name, value in start.items()}
    velocity = {name: np.zeros_like(params[name]) for name in trainable}
    rng = np.random.default_rng(seed)
    x, y = train_set
    history = []
    for epoch in range(epochs + 1):
        if epoch % 5 == 0 or epoch == epochs:
            history.append((epoch, *measures(params, x, y), *measures(params, *valid_set)))
        if epoch == epochs:
            break
        for ids in np.array_split(rng.permutation(len(y)), int(np.ceil(len(y) / 12))):
            logits, cache = forward(x[ids], params)
            grads = backward(y[ids], logits, cache, params)
            for name in trainable:
                velocity[name] = 0.8 * velocity[name] + grads[name]
                params[name] -= rate * velocity[name]
    return params, np.asarray(history)

def move_right(images, pixels, fill=0.0):
    result = np.full_like(images, fill)
    if pixels == 0:
        return images.copy()
    result[..., pixels:] = images[..., :-pixels]
    return result

In [ ]:
if pool_ok:
    source, curve = train_model(initialize(), source_train, source_val, ("K","b","v","c"), epochs=35, rate=.16)
    # History columns: epoch, training loss, training accuracy, validation loss, validation accuracy.
    print("epoch / training BCE / training accuracy / validation BCE / validation accuracy")
    print(curve[[0, 1, -1]])
    print("Actual parameter count:", sum(v.size for v in source.values()))
    np.testing.assert_equal(sum(v.size for v in source.values()), 67)
    plt.plot(curve[:,0], curve[:,1], "o-", label="Training")
    plt.plot(curve[:,0], curve[:,3], "s--", label="Validation")
    plt.xlabel("Epoch")
    plt.ylabel("Binary cross-entropy")
    plt.legend()
    plt.tight_layout()
    plt.show()
    raw = source_val[0]
    moved = move_right(raw, 1)
    old_scores, old_cache = forward(raw, source)
    new_scores, new_cache = forward(moved, source)
    original_map, shifted_map = old_cache[1], new_cache[1]
    # Compare aligned interior positions: original column j becomes moved column j+1.
    aligned_gap = np.max(np.abs(original_map[:, :, :, :-1] - shifted_map[:, :, :, 1:]))
    same_decision = np.mean((old_scores >= 0) == (new_scores >= 0))
    mean_change = np.mean(np.abs(sigmoid(old_scores) - sigmoid(new_scores)))
    print(f"Max aligned conv-map difference: {aligned_gap:.6f}")
    print(f"Paired class agreement: {same_decision:.3f}; mean absolute probability change: {mean_change:.5f}")
    print(f"Accuracy, original = {np.mean((old_scores>=0)==source_val[1]):.3f}, moved = {np.mean((new_scores>=0)==source_val[1]):.3f}")
    fig, ax = plt.subplots(1, 3, figsize=(7, 2.5), constrained_layout=True)
    ch = int(np.argmax(np.std(original_map[0], axis=(1,2))))
    lo, hi = min(original_map[0,ch].min(), shifted_map[0,ch].min()), max(original_map[0,ch].max(), shifted_map[0,ch].max())
    for a, item, title, cmap, low, high in [
        (ax[0], raw[0], "Camera A tile", "gray", 0, 1),
        (ax[1], original_map[0,ch], "Original map", "cividis", lo, hi),
        (ax[2], shifted_map[0,ch], "Map after shift", "cividis", lo, hi)]:
        a.imshow(item, cmap=cmap, vmin=low, vmax=high, interpolation="nearest")
        a.set_title(title, fontsize=10)
        a.axis("off")
    plt.show()
else:
    print("Complete pool2x2 in C3, then run the training experiment.")

**Record your evidence**

Explain the distinction between an equivariant feature map and an invariant output in one sentence each. Use the three printed shift statistics and distinguish label agreement from accuracy. How did BCE change after accuracy first reached its maximum?

*Your answer:*

## C5 · Adapt to the second camera
Camera A saw bright scratches on a dark surface. Camera B sees dark scratches on a lighter, noisier surface. You have 24 labeled target training originals, 48 separate validation originals and a reserved test sample. A left-right mirror preserves a horizontal/vertical orientation label; a 90° turn swaps it. Add a mirrored copy of each **training** tile. The two transfer runs get the same 24 originals, 24 mirrors, fresh binary head, source filters learned in C4, batch order, learning rate and number of epochs.

A **frozen** run updates only the head. A **fine-tuned** run also updates the convolutional filters and biases. Parameter names are K (kernels), b (filter biases), v (head weights), c (head bias). Complete the function to select the trainable names. Choose a run using the lower final **validation BCE**. Accuracy can tie. If the validation losses tie exactly, use the frozen run.

In [ ]:
def fresh_target_start(source):
    start = {name: value.copy() for name, value in source.items()}
    new_head = initialize(seed=19, channels=len(source["b"]))
    start["v"], start["c"] = new_head["v"], new_head["c"]
    return start

def trainable_names(mode):
    # TODO: return the tuple of names to update in each mode.
    # Modes are "frozen" and "fine-tuned".
    return None

In [ ]:
transfer_ok = trainable_names("frozen") is not None and trainable_names("fine-tuned") is not None
if transfer_ok:
    assert set(trainable_names("frozen")) == {"v","c"}
    assert set(trainable_names("fine-tuned")) == {"K","b","v","c"}
if pool_ok and transfer_ok:
    tx, ty = target_train
    x_aug = np.concatenate([tx, np.flip(tx, axis=2)])
    y_aug = np.concatenate([ty, ty])
    start = fresh_target_start(source)
    candidates, valid_losses = {}, {}
    for mode in ("frozen","fine-tuned"):
        fitted, history = train_model(start, (x_aug,y_aug), target_val, trainable_names(mode), epochs=30, rate=.075, seed=44)
        candidates[mode] = fitted
        _, train_loss, train_acc, valid_loss, valid_acc = history[-1]
        valid_losses[mode] = valid_loss
        change = np.linalg.norm(fitted["K"] - start["K"])
        print(f"{mode:10}  train BCE {train_loss:.4f}  validation BCE {valid_loss:.4f}  validation accuracy {valid_acc:.3f}  kernel change {change:.4f}")
    assert np.array_equal(candidates["frozen"]["K"], start["K"])
else:
    print("Complete pool2x2 and trainable_names to run transfer learning.")

**Record your evidence**

Count trainable parameters in both runs. Use validation evidence to choose a run before opening the test cell. Why do we split the original tiles before making mirrors? What label should a 90° rotation of a vertical scratch receive?

*Your answer:*

In [ ]:
if pool_ok and transfer_ok:
    selected = min(valid_losses, key=lambda name: (valid_losses[name], name != "frozen"))
    final_x, final_y = make_tiles(80, 4185, "target")
    test_loss, test_acc = measures(candidates[selected], final_x, final_y)
    print(f"Locked run: {selected}  |  Camera B test BCE: {test_loss:.4f}  |  accuracy: {test_acc:.3f}  |  n={len(final_y)}")
else:
    print("Select the run using validation in C5 before opening the test sample.")

**Record your evidence**

Report the final test loss and accuracy. Name one thing about scratches on a real surface that this dataset cannot test. Save your notebook with the patch calculation, shared gradient, shape audit, paired maps and transfer comparison.

*Your answer:*

### Optional preview · When the border cuts off the evidence
Change the shift from 1 pixel to 8. Some scratches leave the frame. Compare the class agreement again. Inspect a full/cropped pair where the prediction changes, or inspect the largest probability change if all decisions agree. Decide whether a human could still recover the source label. For this comparison report paired agreement; a clipped mark can make the inherited label ambiguous.

In [ ]:
if pool_ok:
    cropped = move_right(source_val[0], 8)
    crop_scores, _ = forward(cropped, source)
    changes = np.abs(sigmoid(crop_scores) - sigmoid(old_scores))
    disagreed = np.flatnonzero((crop_scores >= 0) != (old_scores >= 0))
    idx = int(disagreed[np.argmax(changes[disagreed])]) if len(disagreed) else int(np.argmax(changes))
    print(f"8-pixel paired agreement: {np.mean((crop_scores>=0)==(old_scores>=0)):.3f}; example index: {idx}")
    fig, axes = plt.subplots(1, 2, figsize=(4, 2.2), constrained_layout=True)
    for ax, tile, title in zip(axes, [source_val[0][idx],cropped[idx]], ["Original", "Shifted by 8"]):
        ax.imshow(tile, cmap="gray", vmin=0, vmax=1, interpolation="nearest")
        ax.set_title(title)
        ax.axis("off")
    plt.show()
else:
    print("Train the source network in C4 to try this extension.")